# 데이터 불러오기 및 기초 확인

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [74]:
DATA = '/Users/apple/Documents/team4-final-project/data/interim/'

uqr  = pd.read_csv(DATA + 'accounts_userquestionrecord_preprocessed.csv')
pay  = pd.read_csv(DATA + 'accounts_paymenthistory_preprocessed.csv')
user = pd.read_csv(DATA + 'accounts_user_preprocessed.csv')

paid_users = set(pay['user_id'])
total = len(user)

# ============================================================

# [A] 메인 퍼널 — 전수 point 기준 (이름 정정)

# ============================================================

r = uqr.groupby('chosen_user_id').agg(
받은투표=('id', 'size'),
열람횟수=('opened_times', 'sum')
)
r['point'] = r.index.map(user.set_index('id')['point'])
r['결제']  = r.index.isin(paid_users)

print("=" * 60)
print("[A] 수익 경로 퍼널 (전수 point 기준)")
print("=" * 60)

steps = [
('① 투표 받음',           r),
('② 힌트 열람',           r[r['열람횟수'] > 0]),
('③ 현재 잔액 100 미만',   r[(r['열람횟수'] > 0) & (r['point'] < 100)]),
('④ 결제',               r[(r['열람횟수'] > 0) & (r['point'] < 100) & r['결제']]),
]

prev = None

for name, df in steps:
    n = len(df)

    if prev is None:
        rate = ""
    elif prev == 0:
        rate = "  비율 계산 불가(분모 0)"
    else:
        rate = f"  직전 집단 대비 비율 {n / prev * 100:5.1f}%"

    print(f"{name:24s} {n:>7,}명{rate}")

    prev = n

# ============================================================

# [B] hackle 교차 검증 — 시점별 최저 잔액

# ============================================================

print("\n" + "=" * 60)
print("[B] hackle 시점 추적 (24일간 최저 잔액)")
print("=" * 60)

he = pd.read_csv(DATA + 'hackle_events_preprocessed.csv',
usecols=['session_id', 'event_datetime', 'event_key', 'heart_balance'])
hp = pd.read_csv(DATA + 'hackle_properties_preprocessed.csv',
usecols=['session_id', 'user_id']).drop_duplicates()

he = he.merge(hp, on='session_id')
he = he[he['heart_balance'].notna() & (he['heart_balance'] < 100000)]  # 이상치 제외

bal = he.groupby('user_id')['heart_balance'].agg(
최저='min', 최고='max', 평균='mean'
)
bal['변동폭'] = bal['최고'] - bal['최저']
bal['결제'] = bal.index.isin(paid_users)

print(f"대상: {len(bal):,}명 (전체의 {len(bal)/total*100:.1f}%)")
print()
print(bal[['최저', '최고', '변동폭']].describe().round(0))

# ============================================================

# [B-1] 최저 잔액 구간별 결제율

# ============================================================

print("\n" + "=" * 60)
print("[B-1] 24일간 최저 잔액별 결제율")
print("=" * 60)

print(bal.groupby(pd.cut(bal['최저'], [-1, 99, 299, 999, float('inf')],
labels=['0-99', '100-299', '300-999', '1000+']), observed=True).agg(
인원=('결제', 'size'),
비중=('결제', lambda x: round(len(x)/len(bal)*100, 1)),
결제자=('결제', 'sum'),
결제율=('결제', lambda x: round(x.mean()*100, 2))
))

# ============================================================

# [B-2] 한 번이라도 100 아래로 떨어진 비율

# ============================================================

print("\n" + "=" * 60)
print("[B-2] 소진 경험 여부")
print("=" * 60)

bal['소진경험'] = bal['최저'] < 100
print(f"24일간 잔액이 100 아래로 떨어진 적 있음: "
f"{bal['소진경험'].sum():,}명 ({bal['소진경험'].mean()*100:.1f}%)")
print()
print(bal.groupby('소진경험').agg(
인원=('결제', 'size'),
결제율=('결제', lambda x: round(x.mean()*100, 2)),
평균변동폭=('변동폭', 'mean')
).round(1))

# ============================================================

# [B-3] A와 B 교차 — 두 기준의 일치도

# ============================================================

print("\n" + "=" * 60)
print("[B-3] A(현재 잔액) vs B(최저 잔액) 일치도")
print("=" * 60)

cmp = user[['id', 'point']].set_index('id')
cmp['현재100미만'] = cmp['point'] < 100
cmp['최저100미만'] = cmp.index.map(bal['소진경험'])
cmp = cmp[cmp['최저100미만'].notna()]

print(pd.crosstab(cmp['현재100미만'], cmp['최저100미만'],
rownames=['현재 잔액 <100'], colnames=['24일 최저 <100']))

[A] 수익 경로 퍼널 (전수 point 기준)
① 투표 받음                   15,426명
② 힌트 열람                    7,502명  직전 집단 대비 비율  48.6%
③ 현재 잔액 100 미만               474명  직전 집단 대비 비율   6.3%
④ 결제                          72명  직전 집단 대비 비율  15.2%

[B] hackle 시점 추적 (24일간 최저 잔액)
대상: 217,353명 (전체의 32.1%)

             최저        최고       변동폭
count  217353.0  217353.0  217353.0
mean     2038.0    2149.0     112.0
std      2566.0    2647.0     490.0
min         0.0       0.0       0.0
25%       422.0     512.0       0.0
50%      1166.0    1289.0       0.0
75%      2721.0    2827.0       0.0
max     95474.0   98780.0   36161.0

[B-1] 24일간 최저 잔액별 결제율
             인원    비중    결제자    결제율
최저                                 
0-99      12133   5.6   2705  22.29
100-299   26994  12.4   4885  18.10
300-999   60589  27.9   8060  13.30
1000+    117637  54.1  10184   8.66

[B-2] 소진 경험 여부
24일간 잔액이 100 아래로 떨어진 적 있음: 12,133명 (5.6%)

           인원   결제율  평균변동폭
소진경험                      
False  205220  11.3  104.7
True    12133  22

In [71]:
df_accounts_friendrequest = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/accounts_friendrequest_preprocessed.csv")
df_accounts_friendrequest.head()

,id,status,created_at,updated_at,receive_user_id,send_user_id
0,7,P,2023-04-17 18:29:11,2023-04-17 18:29:11,831962,837521
1,10,A,2023-04-17 18:29:11,2023-04-22 06:02:53,832151,837521
2,11,A,2023-04-17 18:29:11,2023-04-18 19:28:41,832340,837521
3,13,A,2023-04-17 18:29:11,2023-04-19 11:05:04,833041,837521
4,20,P,2023-04-17 18:29:11,2023-04-17 18:29:11,834415,837521


In [3]:
df_friendrequest.info()

<class 'pandas.DataFrame'>
RangeIndex: 17147175 entries, 0 to 17147174
Data columns (total 6 columns):
 #   Column           Dtype
---  ------           -----
 0   id               int64
 1   status           str  
 2   created_at       str  
 3   updated_at       str  
 4   receive_user_id  int64
 5   send_user_id     int64
dtypes: int64(3), str(3)
memory usage: 784.9 MB


In [4]:
df_friendrequest.shape

(17147175, 6)

In [68]:
df_accounts_paymenthistory = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/accounts_paymenthistory_preprocessed.csv")
df_paymenthistory.head()

,id,productId,phone_type,created_at,user_id
0,6,heart.777,A,2023-05-13 21:28:34,1211127
1,7,heart.777,A,2023-05-13 21:29:39,1151343
2,8,heart.777,A,2023-05-13 21:31:33,1002147
3,9,heart.777,A,2023-05-13 21:31:39,1095040
4,11,heart.777,A,2023-05-13 21:34:32,1164081


In [6]:
df_paymenthistory['phone_type'].value_counts()

phone_type
I    61632
A    33508
Name: count, dtype: int64

In [7]:
df_paymenthistory.info()

<class 'pandas.DataFrame'>
RangeIndex: 95140 entries, 0 to 95139
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          95140 non-null  int64
 1   productId   95140 non-null  str  
 2   phone_type  95140 non-null  str  
 3   created_at  95140 non-null  str  
 4   user_id     95140 non-null  int64
dtypes: int64(2), str(3)
memory usage: 3.6 MB


In [8]:
df_paymenthistory.shape 

(95140, 5)

In [9]:
df_paymenthistory.groupby('user_id')['productId'].nunique()

user_id
833041     1
834112     1
835815     2
835839     1
835888     2
          ..
1583408    1
1583445    1
1583450    1
1583454    1
1583632    1
Name: productId, Length: 59192, dtype: int64

In [69]:
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(frame):
        print(frame.to_string(index=False))

# 아래 3개 데이터프레임을 이미 불러온 상태에서 실행합니다.
# df_accounts_userquestionrecord / df_accounts_user / df_accounts_paymenthistory
# 원본 데이터프레임과 앞서 실행한 코드의 r, paid_users, total을 변경하지 않습니다.


def rv_ids(series):
    values = series.astype("string").str.strip()
    values = values.str.replace(r"^([0-9]+)\.0$", r"\1", regex=True)
    return values.replace("", pd.NA)


def rv_pct(n, d):
    return f"{n / d * 100:.2f}%" if d > 0 else "계산 불가(분모 0)"


# 공통 준비: 필요한 열만 복사하고 ID와 숫자 형식을 정리합니다.
rv_user = df_accounts_user[["id", "point"]].copy()
rv_votes = df_accounts_userquestionrecord[["id", "chosen_user_id", "opened_times"]].copy()
rv_user["id"] = rv_ids(rv_user["id"])
rv_user["point"] = pd.to_numeric(rv_user["point"], errors="coerce").astype("Float64")
rv_user["point"] = rv_user["point"].where(~rv_user["point"].isin([float("inf"), -float("inf")]))
rv_user = rv_user.dropna(subset=["id"])

# 같은 회원 ID에 서로 다른 포인트가 있으면 해당 ID의 포인트는 미확인 처리합니다.
rv_conflicts = rv_user.groupby("id")["point"].nunique(dropna=False).gt(1)
rv_user = rv_user.drop_duplicates(subset=["id"]).set_index("id")
rv_user.loc[rv_conflicts[rv_conflicts].index, "point"] = pd.NA
rv_all_users = set(rv_user.index)
rv_total = len(rv_all_users)
rv_paid = set(rv_ids(df_accounts_paymenthistory["user_id"]).dropna()) & rv_all_users

for column in ["id", "chosen_user_id"]:
    rv_votes[column] = rv_ids(rv_votes[column])

if rv_votes["id"].isna().any() or rv_votes["id"].duplicated().any():
    raise ValueError("투표 기록 id에 결측 또는 중복이 있습니다. 건수 집계 전 확인이 필요합니다.")

rv_receivers_all = set(rv_votes["chosen_user_id"].dropna())
rv_missing_receiver = int(rv_votes["chosen_user_id"].isna().sum())
# 이후 [1]~[5]의 수신자와 투표 기록은 모두 회원 테이블과 연결되는 범위입니다.
rv_votes = rv_votes.loc[rv_votes["chosen_user_id"].isin(rv_all_users)].copy()
rv_open = pd.to_numeric(rv_votes["opened_times"], errors="coerce").astype("Float64")
# 횟수이므로 결측·변환 실패·음수·소수·무한대는 미확인 값으로 처리합니다.
rv_valid_open = (rv_open.ge(0) & rv_open.mod(1).eq(0)).fillna(False)
rv_votes["opened_times"] = rv_open.where(rv_valid_open)
rv_votes["열람건"] = rv_votes["opened_times"].gt(0).fillna(False).astype(int)
rv_votes["열람미확인건"] = rv_votes["opened_times"].isna().astype(int)

receiver_summary = rv_votes.groupby("chosen_user_id").agg(
    받은투표=("id", "size"),
    확인된열람횟수=("opened_times", lambda x: x.sum(min_count=1)),
    열람건수=("열람건", "sum"),
    열람미확인건수=("열람미확인건", "sum"),
)
receiver_summary["point"] = rv_user["point"].reindex(receiver_summary.index)
receiver_summary["결제기록"] = receiver_summary.index.isin(rv_paid)
# 양수 기록이 하나라도 있으면 열람 확인. 양수 없이 미확인 값이 있으면 판단 보류.
receiver_summary["열람여부"] = receiver_summary["열람건수"].gt(0).astype("boolean")
rv_unknown_open = (
    receiver_summary["열람건수"].eq(0)
    & receiver_summary["열람미확인건수"].gt(0)
)
receiver_summary.loc[rv_unknown_open, "열람여부"] = pd.NA

# [1] 대상 및 기본 분포
print("[1] 분석 대상: 회원 테이블과 연결되는 투표 수신자")
print(f"회원 고유 사용자: {rv_total:,}명")
print(f"전체 수신 기록의 고유 수신자: {len(rv_receivers_all):,}명")
print(f"회원 테이블과 미연결 수신자: {len(rv_receivers_all - rv_all_users):,}명")
print(f"수신자 ID가 없는 투표 기록: {rv_missing_receiver:,}건")
print(f"분석 수신자: {len(receiver_summary):,}명 ({rv_pct(len(receiver_summary), rv_total)})")
print(f"회원 ID별 포인트 충돌: {int(rv_conflicts.sum()):,}명 → 해당 포인트 미확인 처리")
print(f"분석 기록 중 opened_times 미확인: {int(rv_votes['열람미확인건'].sum()):,}건")
print(f"분석 수신자 중 열람 여부 미확인: {int(rv_unknown_open.sum()):,}명")
print(f"분석 수신자 중 포인트 미확인: {int(receiver_summary['point'].isna().sum()):,}명")
if len(receiver_summary):
    display(receiver_summary[["받은투표", "확인된열람횟수", "point"]].describe().round(1))
print("※ 확인된열람횟수는 유효한 값의 합입니다. 일부 값이 없으면 실제 합계보다 작을 수 있습니다.")

# [2] 조건별 중첩 집계 — 시간 순서가 없는 기록 유무 비교
print("\n[2] 수신·열람·저장 포인트·결제 기록의 조건별 집계")
rv_has_open = receiver_summary["열람여부"].fillna(False)
rv_low_point = receiver_summary["point"].lt(100).fillna(False)
rv_masks = [
    pd.Series(True, index=receiver_summary.index),
    rv_has_open,
    rv_has_open & rv_low_point,
    rv_has_open & rv_low_point & receiver_summary["결제기록"],
]
rv_names = [
    "① 투표 수신 기록 있음", "② 위 집단 중 열람 기록 확인",
    "③ 위 집단 중 저장 포인트 100 미만", "④ 위 집단 중 결제 기록 있음",
]
rv_counts = [int(mask.sum()) for mask in rv_masks]
rv_rows = []
for i, (name, n) in enumerate(zip(rv_names, rv_counts)):
    rv_rows.append({
        "조건": name, "사용자 수": n,
        "직전 집단 대비 비율": "-" if i == 0 else rv_pct(n, rv_counts[i - 1]),
        "전체 회원 대비 비율": rv_pct(n, rv_total),
    })
receiver_conditions = pd.DataFrame(rv_rows)
display(receiver_conditions)
print("※ 100은 제안 코드의 임시 기준입니다. 하트 소진이나 결제 전 잔액을 뜻하지 않습니다.")

# [3] 다음 조건에 포함되지 않는 규모 — 서비스 이탈 인원 아님
print("\n[3] 다음 조건에 포함되지 않는 사용자")
rv_unknown_counts = [
    int(rv_unknown_open.sum()),
    int((rv_has_open & receiver_summary["point"].isna()).sum()),
    0,
]
receiver_exclusions = pd.DataFrame([
    {
        "비교": f"{i + 1}번 조건 → {i + 2}번 조건",
        "다음 집단에 미포함": rv_counts[i] - rv_counts[i + 1],
        "직전 집단 대비 비율": rv_pct(rv_counts[i] - rv_counts[i + 1], rv_counts[i]),
        "그중 열람·포인트 값 미확인": rv_unknown_counts[i],
    }
    for i in range(3)
])
display(receiver_exclusions)
print("※ 조건 미충족 또는 값 미확인 인원입니다. 서비스 이탈이나 다음 행동 포기를 입증하지 않습니다.")

# [4] opened_times 임계값별 집계: 사용자 수가 아니라 투표 기록 건수
print("\n[4] 열어본 횟수별 투표 기록 — 힌트 단계 수는 미확인")
rv_open_valid = rv_votes["opened_times"].dropna()
rv_open_rows = []
rv_previous = None
for threshold in [1, 2, 3]:
    n = int(rv_open_valid.ge(threshold).sum())
    rv_open_rows.append({
        "조건": f"opened_times ≥ {threshold}", "투표 기록 수": n,
        "유효한 전체 기록 대비 비율": rv_pct(n, len(rv_open_valid)),
        "직전 조건 대비 비율": "-" if rv_previous is None else rv_pct(n, rv_previous),
    })
    rv_previous = n
receiver_open_counts = pd.DataFrame(rv_open_rows)
display(receiver_open_counts)
print(f"유효한 opened_times 기록: {len(rv_open_valid):,}건")
if len(rv_open_valid):
    print(f"관측 최댓값: {rv_open_valid.max():g} / 3 초과 기록: {int(rv_open_valid.gt(3).sum()):,}건")
else:
    print("관측 최댓값: 확인 불가(유효한 값 없음)")

# [5] 받은 투표 구간별 비교 — 각 비율은 병렬 지표이며 순차 전환율이 아닙니다.
print("\n[5] 받은 투표 구간별 기록·포인트 비교")
receiver_summary["구간"] = pd.cut(
    receiver_summary["받은투표"], [0, 5, 20, 100, 500, float("inf")],
    labels=["1~5", "6~20", "21~100", "101~500", "501 이상"],
)
rv_segment_rows = []
for label, group in receiver_summary.groupby("구간", observed=True):
    open_known = group["열람여부"].notna()
    point_known = group["point"].notna()
    rv_segment_rows.append({
        "받은 투표 구간": str(label), "인원": len(group),
        "열람 판단 가능 인원": int(open_known.sum()),
        "열람 기록 보유율": rv_pct(int(group["열람여부"].fillna(False).sum()), int(open_known.sum())),
        "포인트 확인 인원": int(point_known.sum()),
        "포인트 100 미만 비율": rv_pct(int(group["point"].lt(100).fillna(False).sum()), int(point_known.sum())),
        "결제 기록 보유율": rv_pct(int(group["결제기록"].sum()), len(group)),
    })
receiver_segments = pd.DataFrame(rv_segment_rows)
display(receiver_segments)
print("※ 열람·포인트 비율의 분모는 각 값을 판단할 수 있는 인원, 결제 기록 보유율의 분모는 구간 전체 인원입니다.")
print("※ [2]는 직전 집단 전체가 분모이므로, 미확인 값이 있으면 [5]의 비율과 다를 수 있습니다.")
print("※ 분석 기간, 행동 순서, 관찰 기간, 첫 구매 여부는 반영하지 않았습니다.")


[1] 분석 대상: 회원 테이블과 연결되는 투표 수신자
회원 고유 사용자: 677,080명
전체 수신 기록의 고유 수신자: 15,426명
회원 테이블과 미연결 수신자: 0명
수신자 ID가 없는 투표 기록: 0건
분석 수신자: 15,426명 (2.28%)
회원 ID별 포인트 충돌: 0명 → 해당 포인트 미확인 처리
분석 기록 중 opened_times 미확인: 0건
분석 수신자 중 열람 여부 미확인: 0명
분석 수신자 중 포인트 미확인: 0명


,받은투표,확인된열람횟수,point
count,15426.0,15426.0,15426.0
mean,78.9,5.0,1825.7
std,131.4,9.4,2176.7
min,1.0,0.0,0.0
25%,3.0,0.0,471.0
50%,15.0,0.0,1150.0
75%,97.0,6.0,2438.8
max,1239.0,101.0,58990.0


※ 확인된열람횟수는 유효한 값의 합입니다. 일부 값이 없으면 실제 합계보다 작을 수 있습니다.

[2] 수신·열람·저장 포인트·결제 기록의 조건별 집계


,조건,사용자 수,직전 집단 대비 비율,전체 회원 대비 비율
0,① 투표 수신 기록 있음,15426,-,2.28%
1,② 위 집단 중 열람 기록 확인,7502,48.63%,1.11%
2,③ 위 집단 중 저장 포인트 100 미만,474,6.32%,0.07%
3,④ 위 집단 중 결제 기록 있음,72,15.19%,0.01%


※ 100은 제안 코드의 임시 기준입니다. 하트 소진이나 결제 전 잔액을 뜻하지 않습니다.

[3] 다음 조건에 포함되지 않는 사용자


...

※ 조건 미충족 또는 값 미확인 인원입니다. 서비스 이탈이나 다음 행동 포기를 입증하지 않습니다.

[4] 열어본 횟수별 투표 기록 — 힌트 단계 수는 미확인


,조건,투표 기록 수,유효한 전체 기록 대비 비율,직전 조건 대비 비율
0,opened_times ≥ 1,60662,4.98%,-
1,opened_times ≥ 2,13780,1.13%,22.72%
2,opened_times ≥ 3,2660,0.22%,19.30%


유효한 opened_times 기록: 1,217,558건
관측 최댓값: 3 / 3 초과 기록: 0건

[5] 받은 투표 구간별 기록·포인트 비교


,받은 투표 구간,인원,열람 판단 가능 인원,열람 기록 보유율,포인트 확인 인원,포인트 100 미만 비율,결제 기록 보유율
0,1~5,5122,5122,13.00%,5122,5.06%,7.87%
1,6~20,3361,3361,32.49%,3361,4.37%,7.38%
2,21~100,3148,3148,65.95%,3148,5.30%,6.86%
3,101~500,3492,3492,96.42%,3492,5.10%,8.93%
4,501 이상,303,303,99.34%,303,4.95%,11.22%


※ 열람·포인트 비율의 분모는 각 값을 판단할 수 있는 인원, 결제 기록 보유율의 분모는 구간 전체 인원입니다.
※ [2]는 직전 집단 전체가 분모이므로, 미확인 값이 있으면 [5]의 비율과 다를 수 있습니다.
※ 분석 기간, 행동 순서, 관찰 기간, 첫 구매 여부는 반영하지 않았습니다.


In [72]:
"""친구 요청 발신 → 투표 수신 → 후속 결제의 관측 경로를 분석합니다.

기존 사용자의 관계 형성 시도를 출발점으로 삼습니다. 수락 시점은 확인되지
않았으므로 요청의 created_at을 사용하고, 이후 chosen_user_id로 수신자를
연결합니다. 직접 투표하거나 요청을 보내야만 수신할 수 있다는 뜻은 아닙니다.
결과의 적용 대상은 선택 기간의 친구 요청 발신자입니다.

created_at이 각 행동 시점이라는 전제, 추출 이력의 완전성, 기간과 시간대를
확인해야 합니다. 7일 + 7일은 수정 가능한 분석 설정이며 확정된 서비스 기준이
아닙니다. 후속 결제에는 첫 구매와 재구매가 모두 포함될 수 있습니다.
"""

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(frame):
        print(frame.to_string(index=False))

# 이미 불러온 데이터프레임을 사용합니다. 원본을 변경하지 않습니다.
# df_accounts_friendrequest: send_user_id, created_at
# df_accounts_userquestionrecord: chosen_user_id, created_at
# df_accounts_paymenthistory: user_id, created_at
# df_accounts_user: id

# 팀이 정한 기간이 있으면 None 대신 시각까지 지정하세요.
# None이면 세 테이블의 관측 시각 범위가 겹치는 구간을 임시로 사용합니다.
# 자동 기간은 기록의 완전성이나 동일한 추출 모집단을 보장하지 않습니다.
ANALYSIS_START = None
OBSERVATION_END = None
RECEIVE_WITHIN_DAYS = 7
PAY_WITHIN_DAYS = 7
NAIVE_TIMEZONE = "Asia/Seoul"  # 시간대 없는 전처리 날짜는 이미 KST라는 전제


def build_receiver_time_funnel(
    friendrequest, userquestionrecord, paymenthistory, users,
    analysis_start=None, observation_end=None,
    receive_days=7, pay_days=7, naive_timezone="Asia/Seoul",
):
    """기간 내 첫 발신 → 첫 후속 수신 → 첫 후속 결제. 생애 첫 행동을 뜻하지 않음."""
    if receive_days <= 0 or pay_days <= 0:
        raise ValueError("두 관찰 기간은 모두 0보다 커야 합니다.")

    def clean_id(series):
        values = series.astype("string").str.strip()
        return values.str.replace(r"^([0-9]+)\.0$", r"\1", regex=True).replace("", pd.NA)

    def parse_time(series):
        # 이미 시간대가 있으면 변환만 하고, 없는 값에는 지정 시간대를 부여합니다.
        # UTC 변환이나 +9시간을 일괄 적용하지 않습니다.
        try:
            parsed = pd.to_datetime(series, errors="coerce", format="mixed")
            if parsed.dt.tz is None:
                parsed = parsed.dt.tz_localize(naive_timezone)
            else:
                parsed = parsed.dt.tz_convert(naive_timezone)
            return parsed.dt.as_unit("ns")
        except (AttributeError, TypeError, ValueError) as exc:
            raise ValueError("날짜 열에 서로 다른 시간대 형식이 섞였는지 확인하세요.") from exc

    def boundary(value):
        timestamp = pd.Timestamp(value)
        return (
            timestamp.tz_localize(naive_timezone)
            if timestamp.tzinfo is None else timestamp.tz_convert(naive_timezone)
        ).as_unit("ns")

    def percent(n, d):
        return f"{n / d * 100:.2f}%" if d > 0 else "계산 불가(분모 0)"

    member_ids = set(clean_id(users["id"]).dropna())
    source_rows = []

    def prepare(frame, actor_column, name):
        required = [actor_column, "created_at"]
        missing = [column for column in required if column not in frame.columns]
        if missing:
            raise KeyError(f"{name}에 {missing} 열이 없습니다. 해당 열을 포함한 데이터프레임이 필요합니다.")
        events = pd.DataFrame({
            "user_id": clean_id(frame[actor_column]),
            "event_at": parse_time(frame["created_at"]),
        })
        missing_id = events["user_id"].isna()
        unmatched_id = events["user_id"].notna() & ~events["user_id"].isin(member_ids)
        invalid_time = events["event_at"].isna()
        usable = events.loc[~missing_id & ~unmatched_id & ~invalid_time].copy()
        source_rows.append({
            "기록": name, "원본 행 수": len(events), "유효 행 수": len(usable),
            "ID 결측": int(missing_id.sum()), "회원 미연결": int(unmatched_id.sum()),
            "시각 미확인": int(invalid_time.sum()),
            "최초 유효 시각": usable["event_at"].min(),
            "최종 유효 시각": usable["event_at"].max(),
        })
        return usable

    # A 상태로 제한하지 않습니다. 요청 생성 당시의 발신 행동을 기준으로 합니다.
    sent = prepare(friendrequest, "send_user_id", "친구 요청 발신")
    received = prepare(userquestionrecord, "chosen_user_id", "투표 수신")
    payments = prepare(paymenthistory, "user_id", "결제")
    sources = pd.DataFrame(source_rows)

    if (analysis_start is None or observation_end is None) and any(
        frame.empty for frame in [sent, received, payments]
    ):
        raise ValueError("유효 기록이 없는 테이블이 있어 기간을 자동 설정할 수 없습니다. 기간 또는 원본을 확인하세요.")
    start = boundary(analysis_start) if analysis_start is not None else sources["최초 유효 시각"].max()
    end = boundary(observation_end) if observation_end is not None else sources["최종 유효 시각"].min()
    if pd.isna(start) or pd.isna(end) or start > end:
        raise ValueError("분석 시작·종료 시각이 유효하지 않거나 공통 관측 구간이 없습니다.")

    receive_window = pd.Timedelta(days=receive_days)
    pay_window = pd.Timedelta(days=pay_days)
    entry_cutoff = end - receive_window - pay_window
    event_dtype = pd.DatetimeTZDtype(unit="ns", tz=naive_timezone)

    # 사용자마다 선택한 기간 내 첫 발신을 기준 시점으로 고정합니다.
    sent_in_period = sent.loc[sent["event_at"].between(start, end)]
    entrants = sent_in_period.groupby("user_id")["event_at"].min().rename("request_at")
    mature = entrants.loc[entrants <= entry_cutoff]
    details = mature.to_frame()

    def first_after(anchors, anchor_column, events):
        left = anchors[["user_id", anchor_column]].dropna().sort_values(anchor_column)
        right = events[["user_id", "event_at"]].sort_values("event_at")
        window = receive_window if anchor_column == "request_at" else pay_window
        if left.empty or right.empty:
            return pd.Series(pd.NaT, index=anchors["user_id"], dtype=event_dtype)
        matched = pd.merge_asof(
            left, right, left_on=anchor_column, right_on="event_at", by="user_id",
            direction="forward", allow_exact_matches=False, tolerance=window,
        )
        return matched.set_index("user_id")["event_at"]

    received_in_period = received.loc[received["event_at"].between(start, end)]
    payments_in_period = payments.loc[payments["event_at"].between(start, end)]
    details["received_at"] = first_after(details.reset_index(), "request_at", received_in_period)
    details["paid_at"] = first_after(details.reset_index(), "received_at", payments_in_period)
    details["request_to_receive_hours"] = (details["received_at"] - details["request_at"]).dt.total_seconds() / 3600
    details["receive_to_pay_hours"] = (details["paid_at"] - details["received_at"]).dt.total_seconds() / 3600
    # 전체 불러온 결제 이력으로 이전 결제 기록 유무를 추가 확인합니다.
    first_observed_payment = payments.groupby("user_id")["event_at"].min()
    details["first_observed_payment_at"] = first_observed_payment.reindex(details.index)
    details["has_earlier_payment_record"] = (
        details["paid_at"].notna()
        & details["first_observed_payment_at"].lt(details["paid_at"])
    )

    n1 = len(details)
    n2 = int(details["received_at"].notna().sum())
    n3 = int(details["paid_at"].notna().sum())
    funnel = pd.DataFrame([
        {"단계": "① 기간 내 첫 친구 요청 발신", "사용자 수": n1,
         "직전 단계 대비 전환율": "-", "진입자 대비 비율": percent(n1, n1)},
        {"단계": f"② 발신 후 {receive_days}일 이내 첫 후속 수신", "사용자 수": n2,
         "직전 단계 대비 전환율": percent(n2, n1), "진입자 대비 비율": percent(n2, n1)},
        {"단계": f"③ 위 수신 후 {pay_days}일 이내 첫 후속 결제", "사용자 수": n3,
         "직전 단계 대비 전환율": percent(n3, n2), "진입자 대비 비율": percent(n3, n1)},
    ])
    checks = {
        "analysis_start": start, "observation_end": end, "entry_cutoff": entry_cutoff,
        "member_users": len(member_ids), "entry_users_in_period": len(entrants),
        "excluded_short_followup_users": int((entrants > entry_cutoff).sum()),
        "received_users_outside_entry_cohort": len(set(received_in_period["user_id"]) - set(details.index)),
        "paid_users_outside_completed_path": len(set(payments_in_period["user_id"]) - set(details.index[details["paid_at"].notna()])),
        "converted_with_earlier_payment_record": int(details["has_earlier_payment_record"].sum()),
    }
    return details, funnel, sources, checks


# 실행
temporal_users, temporal_funnel, temporal_sources, temporal_checks = build_receiver_time_funnel(
    df_accounts_friendrequest, df_accounts_userquestionrecord,
    df_accounts_paymenthistory, df_accounts_user,
    analysis_start=ANALYSIS_START, observation_end=OBSERVATION_END,
    receive_days=RECEIVE_WITHIN_DAYS, pay_days=PAY_WITHIN_DAYS,
    naive_timezone=NAIVE_TIMEZONE,
)

print("[1] 원본 기록과 날짜 확인")
display(temporal_sources)
print("※ ID·시각의 문제 건수는 서로 겹칠 수 있습니다.")
print("\n[2] 분석 기준")
print(f"시작 시각: {temporal_checks['analysis_start']}")
print(f"최종 관찰 시각: {temporal_checks['observation_end']}")
print(f"포함 가능한 발신의 마지막 시각: {temporal_checks['entry_cutoff']}")
print(f"관찰 시간 부족으로 제외한 발신자: {temporal_checks['excluded_short_followup_users']:,}명")
if temporal_checks["entry_cutoff"] < temporal_checks["analysis_start"]:
    print("※ 관측 구간이 두 관찰 기간의 합보다 짧아, 분석에 포함할 발신자가 없습니다.")
if ANALYSIS_START is None or OBSERVATION_END is None:
    print("※ 자동 기간은 관측 시각 범위의 교집합입니다. 추출 범위와 기록 완전성은 별도로 확인해야 합니다.")
print("\n[3] 시점을 반영한 수신자 관점의 경로 퍼널")
display(temporal_funnel)
print(f"회원 테이블 고유 사용자: {temporal_checks['member_users']:,}명")
print(f"기간 내 수신자 중 발신 코호트 밖의 사용자: {temporal_checks['received_users_outside_entry_cohort']:,}명")
print(f"기간 내 결제자 중 전체 경로 미충족 사용자: {temporal_checks['paid_users_outside_completed_path']:,}명")
print(f"경로 전환자 중 이전 결제 기록도 있는 사용자: {temporal_checks['converted_with_earlier_payment_record']:,}명")
print("※ 첫 후속 수신·결제는 기준 행동 이후의 첫 기록이며, 생애 첫 수신·구매가 아닙니다.")
print("※ 같은 시각의 기록은 순서를 판별할 수 없어 제외하고, 기한과 정확히 같은 시각은 포함합니다.")
print("※ 수신 시점은 수신 기록의 created_at으로 간주합니다. 실제 생성 조건을 확인해야 합니다.")
print("※ 친구 요청이 수신·결제를 유발했다는 인과관계나, 이 경로가 필수라는 뜻은 아닙니다.")
print("※ opened_times와 현재 point는 행동 시각이 없어 이 순차 퍼널에 넣지 않았습니다.")


[1] 원본 기록과 날짜 확인


,기록,원본 행 수,유효 행 수,ID 결측,회원 미연결,시각 미확인,최초 유효 시각,최종 유효 시각
0,친구 요청 발신,17147175,17147175,0,0,0,2023-04-17 18:29:11+09:00,2024-05-09 09:21:47+09:00
1,투표 수신,1217558,1217558,0,0,0,2023-04-28 12:27:49+09:00,2024-05-08 01:36:18+09:00
2,결제,95140,95140,0,0,0,2023-05-13 21:28:34+09:00,2024-05-08 14:12:45+09:00


※ ID·시각의 문제 건수는 서로 겹칠 수 있습니다.

[2] 분석 기준
시작 시각: 2023-05-13 21:28:34+09:00
최종 관찰 시각: 2024-05-08 01:36:18+09:00
포함 가능한 발신의 마지막 시각: 2024-04-24 01:36:18+09:00
관찰 시간 부족으로 제외한 발신자: 71명
※ 자동 기간은 관측 시각 범위의 교집합입니다. 추출 범위와 기록 완전성은 별도로 확인해야 합니다.

[3] 시점을 반영한 수신자 관점의 경로 퍼널


,단계,사용자 수,직전 단계 대비 전환율,진입자 대비 비율
0,① 기간 내 첫 친구 요청 발신,471559,-,100.00%
1,② 발신 후 7일 이내 첫 후속 수신,6079,1.29%,1.29%
2,③ 위 수신 후 7일 이내 첫 후속 결제,393,6.46%,0.08%


회원 테이블 고유 사용자: 677,080명
기간 내 수신자 중 발신 코호트 밖의 사용자: 5,943명
기간 내 결제자 중 전체 경로 미충족 사용자: 58,798명
경로 전환자 중 이전 결제 기록도 있는 사용자: 31명
※ 첫 후속 수신·결제는 기준 행동 이후의 첫 기록이며, 생애 첫 수신·구매가 아닙니다.
※ 같은 시각의 기록은 순서를 판별할 수 없어 제외하고, 기한과 정확히 같은 시각은 포함합니다.
※ 수신 시점은 수신 기록의 created_at으로 간주합니다. 실제 생성 조건을 확인해야 합니다.
※ 친구 요청이 수신·결제를 유발했다는 인과관계나, 이 경로가 필수라는 뜻은 아닙니다.
※ opened_times와 현재 point는 행동 시각이 없어 이 순차 퍼널에 넣지 않았습니다.


In [10]:
df_user_contacts = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/accounts_user_contacts_preprocessed.csv")
df_user_contacts.head()



,id,contacts_count,invite_user_id_list,user_id
0,259,30,[],1167696
1,1756,79,[],863169
2,13742,21,[854615],857205
3,13754,29,[],851431
4,13756,28,[849318],855476


In [11]:
df_user_contacts.info()

<class 'pandas.DataFrame'>
RangeIndex: 5063 entries, 0 to 5062
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   5063 non-null   int64
 1   contacts_count       5063 non-null   int64
 2   invite_user_id_list  5063 non-null   str  
 3   user_id              5063 non-null   int64
dtypes: int64(3), str(1)
memory usage: 158.3 KB


In [12]:
df_user_contacts.shape

(5063, 4)

In [62]:
df_accounts_user = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/accounts_user_preprocessed.csv")

df_accounts_user.head()

,id,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,831962,F,2248,1,2023-03-29 05:18:56.162368,N,253,40878,5499,110,12.0
1,832151,M,1519,0,2023-03-29 12:56:34.989468,N,0,37,0,47,1.0
2,832340,F,57,1,2023-03-29 12:56:35.020790,N,0,19,0,21,1.0
3,832520,M,1039,0,2023-03-29 12:56:35.049311,N,0,29,0,15,12.0
4,832614,M,1048,1,2023-03-29 12:56:35.064406,N,0,28,0,14,12.0


In [14]:
df_user.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             677080 non-null  int64  
 1   gender         677080 non-null  str    
 2   point          677080 non-null  int64  
 3   is_push_on     677080 non-null  int64  
 4   created_at     677080 non-null  str    
 5   ban_status     677080 non-null  str    
 6   report_count   677080 non-null  int64  
 7   alarm_count    677080 non-null  int64  
 8   pending_chat   677080 non-null  int64  
 9   pending_votes  677080 non-null  int64  
 10  group_id       677080 non-null  float64
dtypes: float64(1), int64(7), str(3)
memory usage: 56.8 MB


In [15]:
df_user.shape

(677080, 11)

In [16]:
df_user.groupby('id').nunique()

,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
id,,,,,,,,,,
831962,1,1,1,1,1,1,1,1,1,1
832151,1,1,1,1,1,1,1,1,1,1
832340,1,1,1,1,1,1,1,1,1,1
832520,1,1,1,1,1,1,1,1,1,1
832614,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...
1583729,1,1,1,1,1,1,1,1,1,1
1583730,1,1,1,1,1,1,1,1,1,1
1583731,1,1,1,1,1,1,1,1,1,1


In [65]:
df_accounts_userquestionrecord = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/accounts_userquestionrecord_preprocessed.csv")
df_accounts_userquestionrecord.head()

,id,status,created_at,chosen_user_id,question_id,user_id,question_piece_id,has_read,answer_status,answer_updated_at,report_count,opened_times
0,771777,C,2023-04-28 12:27:49,849469,252,849436,998458,0,N,2023-04-28 12:27:49,0,0
1,771800,C,2023-04-28 12:28:02,849446,244,849436,998459,0,N,2023-04-28 12:28:02,0,0
2,771812,C,2023-04-28 12:28:09,849454,183,849436,998460,1,N,2023-04-28 12:28:09,0,0
3,771828,C,2023-04-28 12:28:16,847375,101,849436,998461,0,N,2023-04-28 12:28:16,0,0
4,771851,C,2023-04-28 12:28:26,849477,209,849436,998462,1,N,2023-04-28 12:28:26,0,0


In [66]:
df_accounts_userquestionrecord.info()

<class 'pandas.DataFrame'>
RangeIndex: 1217558 entries, 0 to 1217557
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   id                 1217558 non-null  int64
 1   status             1217558 non-null  str  
 2   created_at         1217558 non-null  str  
 3   chosen_user_id     1217558 non-null  int64
 4   question_id        1217558 non-null  int64
 5   user_id            1217558 non-null  int64
 6   question_piece_id  1217558 non-null  int64
 7   has_read           1217558 non-null  int64
 8   answer_status      1217558 non-null  str  
 9   answer_updated_at  1217558 non-null  str  
 10  report_count       1217558 non-null  int64
 11  opened_times       1217558 non-null  int64
dtypes: int64(8), str(4)
memory usage: 111.5 MB


In [19]:
df_userquestionrecord.shape

(1217558, 12)

In [20]:
df_events = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/hackle_events_preprocessed.csv")
df_events.head()

,event_id,event_datetime,event_key,session_id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,2023-07-18 19:40:17,$session_start,4OzYh3seq3VKytpSn5pvQkZNQii1,NaN,NaN,NaN,NaN,NaN,NaN
1,00000716-27e9-4e72-a602-d0ce61784b06,2023-07-18 21:07:24,click_question_open,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,NaN,NaN,64.0,436.0,4830.0,NaN
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,2023-08-06 20:18:03,click_bottom_navigation_profile,6bcea65d-9f40-46fc-888c-700fe707483f,NaN,NaN,26.0,174.0,4729.0,NaN
3,00000981-5e2a-4111-993e-4f1891ad9a53,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,NaN,NaN,61.0,44.0,142.0,NaN
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,2023-07-24 15:03:37,click_bottom_navigation_lab,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,NaN,NaN,119.0,545.0,3287.0,NaN


In [21]:
df_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 11441319 entries, 0 to 11441318
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   event_id        str    
 1   event_datetime  str    
 2   event_key       str    
 3   session_id      str    
 4   item_name       str    
 5   page_name       str    
 6   friend_count    float64
 7   votes_count     float64
 8   heart_balance   float64
 9   question_id     float64
dtypes: float64(4), str(6)
memory usage: 872.9 MB


In [22]:
df_events.shape

(11441319, 10)

In [23]:
df_properties = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/hackle_properties_preprocessed.csv")   
df_properties.head()

,id,session_id,user_id,language,osname,osversion,versionname,device_id
0,2,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,1046711,ko-KR,iOS,16.5.1,2.0.3,D5417226-F71B-4A9E-A180-CD072F2AB279
1,3,6bcea65d-9f40-46fc-888c-700fe707483f,1545130,ko,Android,13,2.0.5,6bcea65d-9f40-46fc-888c-700fe707483f
2,4,XVYNT6zfhFWqIg9omwg2AHDjTLx2,1224793,ko,Android,13,2.0.5,a05c1595-3e05-434b-8684-218b528bd725
3,5,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,1329450,ko-US,iOS,16.5.1,2.0.5,EAC6C0B3-7CE8-40EA-8A91-9977C0BA5EF3
4,7,d2b3ca43-4716-4852-b0e2-334848eb66f4,1018001,ko,Android,13,2.0.5,d2b3ca43-4716-4852-b0e2-334848eb66f4


In [24]:
df_properties.info()

<class 'pandas.DataFrame'>
RangeIndex: 327431 entries, 0 to 327430
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           327431 non-null  int64
 1   session_id   327431 non-null  str  
 2   user_id      327431 non-null  int64
 3   language     327431 non-null  str  
 4   osname       327431 non-null  str  
 5   osversion    327431 non-null  str  
 6   versionname  327431 non-null  str  
 7   device_id    327431 non-null  str  
dtypes: int64(2), str(6)
memory usage: 20.0 MB


In [25]:
df_properties.shape

(327431, 8)

In [26]:
df_question = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/polls_question_preprocessed.csv")
df_question.head()

,id,question_text,created_at
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53
2,101,미래의 틱톡커는?,2023-03-31 15:22:54
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55


In [27]:
df_question.info()

<class 'pandas.DataFrame'>
RangeIndex: 5025 entries, 0 to 5024
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             5025 non-null   int64
 1   question_text  5025 non-null   str  
 2   created_at     5025 non-null   str  
dtypes: int64(1), str(2)
memory usage: 117.9 KB


In [28]:
df_question.shape

(5025, 3)

In [29]:
df_questionpiece = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/polls_questionpiece_preprocessed.csv")
df_questionpiece.head()

,id,is_voted,created_at,question_id,is_skipped
0,998458,1,2023-04-28 12:27:22,252,0
1,998459,1,2023-04-28 12:27:22,244,0
2,998460,1,2023-04-28 12:27:22,183,0
3,998461,1,2023-04-28 12:27:22,101,0
4,998462,1,2023-04-28 12:27:22,209,0


In [30]:
df_questionpiece.info()

<class 'pandas.DataFrame'>
RangeIndex: 1264349 entries, 0 to 1264348
Data columns (total 5 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   id           1264349 non-null  int64
 1   is_voted     1264349 non-null  int64
 2   created_at   1264349 non-null  str  
 3   question_id  1264349 non-null  int64
 4   is_skipped   1264349 non-null  int64
dtypes: int64(4), str(1)
memory usage: 48.2 MB


In [31]:
df_questionpiece.shape

(1264349, 5)

In [32]:
df_questionset = pd.read_csv("/Users/apple/Documents/team4-final-project/data/interim/polls_questionset_preprocessed.csv")
df_questionset.head()

,id,question_piece_id_list,opening_time,status,created_at,user_id
0,99830,"[998588, 998589, 998590, 998591, 998592, 99859...",2023-04-28 12:28:07,F,2023-04-28 12:28:07,849438
1,99840,"[998689, 998691, 998693, 998695, 998697, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,847375
2,99841,"[998688, 998690, 998692, 998694, 998696, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446
3,99848,"[998768, 998769, 998770, 998771, 998772, 99877...",2023-04-28 12:28:57,F,2023-04-28 12:28:57,849477
4,99852,"[998808, 998809, 998810, 998811, 998813, 99881...",2023-04-28 12:29:04,F,2023-04-28 12:29:04,849469


In [33]:
df_questionset.info()

<class 'pandas.DataFrame'>
RangeIndex: 157705 entries, 0 to 157704
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   id                      157705 non-null  int64
 1   question_piece_id_list  157705 non-null  str  
 2   opening_time            157705 non-null  str  
 3   status                  157705 non-null  str  
 4   created_at              157705 non-null  str  
 5   user_id                 157705 non-null  int64
dtypes: int64(2), str(4)
memory usage: 7.2 MB


In [34]:
df_questionset.shape

(157705, 6)

# EDA

## 1. 친구 관련 EDA

###  df_friendrequest, df_user 관련 컬럼 사용

In [35]:
friend = df_friendrequest[
    ["id", "send_user_id", "receive_user_id", "created_at"]
].copy()

friend["created_at"] = pd.to_datetime(
    friend["created_at"],
    format="mixed",
    errors="coerce"
)

print("날짜 결측·변환 실패:", friend["created_at"].isna().sum())

날짜 결측·변환 실패: 0


In [ ]:
## ① 친구 요청 기록의 기간 확인 - 이후 질문 행동과 연결하려면, 우선 친구 요청이 어느 기간에 기록되어 있는지 확인
friend["created_at"].agg(["min", "max"])

min   2023-04-17 18:29:11
max   2024-05-09 09:21:47
Name: created_at, dtype: datetime64[us]

-> 대략 데이터 기간은 23년 4월 중순부터 24년 5월 초로 1년 하고도 3주정도의 기간

In [ ]:
## ② 사용자별로 요청을 얼마나 보내고 받았는지 확인 - 전체 요청이 많아도 소수의 사용자가 보내거나 받는 경우가 있을 수 있으므로, 사용자별로 요청을 얼마나 보내고 받았는지 확인

sent_count = friend.groupby("send_user_id")["id"].nunique()
received_count = friend.groupby("receive_user_id")["id"].nunique()

user_friend = df_user[["id"]].drop_duplicates().copy()
user_friend["sent_count"] = (
    user_friend["id"].map(sent_count).fillna(0).astype(int)
)

user_friend["received_count"] = (
    user_friend["id"].map(received_count).fillna(0).astype(int)
)

user_friend[["sent_count", "received_count"]].describe()

,sent_count,received_count
count,677080.000000,677080.000000
mean,25.325183,25.325183
std,23.917493,19.640004
min,0.000000,0.000000
25%,10.000000,10.000000
50%,20.000000,21.000000
75%,34.000000,37.000000
max,1816.000000,218.000000


-> 최대값에 이상치처럼 보이는 값들이 보임 => 1,816건이 하루에 발생했는지, 수개월에 걸쳐 쌓였는지와 같은 추가 검증 필요해보임

In [ ]:
## ②-1. 큰 값을 판별하는 기준과 해당 사용자 수 확인 (IQR 방식)

cols = ["sent_count", "received_count"]

q1 = user_friend[cols].quantile(0.25)
q3 = user_friend[cols].quantile(0.75)

upper = q3 + 1.5 * (q3 - q1)
over = user_friend[cols].gt(upper)

pd.DataFrame({
    "상단 점검 기준": upper,
    "기준 초과 사용자 수": over.sum(),
    "기준 초과 비율(%)": over.mean() * 100
}).round(2)

,상단 점검 기준,기준 초과 사용자 수,기준 초과 비율(%)
sent_count,70.0,28864,4.26
received_count,77.5,11294,1.67


-> 전체 사용자 중 발신 70건 초과는 28,864명(4.26%), 수신 77.5건 초과는 11,294명(1.67%)으로, 요청량이 큰 점검 대상 사용자 확인

In [46]:
## ②-2. 발신자 상위 약 1%가 전체 요청에서 차지하는 비중 확인

from math import ceil

senders_count = user_friend.loc[
    user_friend["sent_count"] > 0, "sent_count"
]

top_n = ceil(len(senders_count) * 0.01)

top_share = (
    senders_count.nlargest(top_n).sum()
    / senders_count.sum()
    * 100
)

print("상위 약 1% 발신자 수:", top_n)
print("이들이 차지하는 요청 비중(%):", round(top_share, 2))

상위 약 1% 발신자 수: 6491
이들이 차지하는 요청 비중(%): 5.61


-> 요청 발신자 상위 약 1%인 6,491명이 전체 발신 요청의 5.61%를 차지

In [42]:
## ③ 요청을 보내는 사람과 받는 사람을 나누어 보기 - 직접 관계를 만들려고 요청하는 사람과 다른 사람에게 요청받는 사람은 이용 형태가 다를 것이라는 생각에서 비롯하여 확인
user_friend["sent_any"] = user_friend["sent_count"] > 0
user_friend["received_any"] = user_friend["received_count"] > 0

pd.crosstab(
    user_friend["sent_any"],
    user_friend["received_any"],
    rownames=["요청을 보냈는가"],
    colnames=["요청을 받았는가"]
)

요청을 받았는가,False,True
요청을 보냈는가,,
False,3375,24633
True,12864,636208


| 보냄    | 받음    | 의미                  |
| ----- | ----- | ------------------- |
| False | False | 보낸 기록도 받은 기록도 없음    |
| True  | False | 보낸 기록만 있음           |
| False | True  | 받은 기록만 있음           |
| True  | True  | 보낸 기록과 받은 기록이 모두 있음 |


-> 보낸 기록과 받은 기록이 모두 있는 사용자가 약 94%로 대부분을 차지

In [ ]:
## ④ 요청 횟수와 요청한 상대 수를 구분하기 - 한 사람이 요청을 10건 보냈다고 해서 반드시 서로 다른 10명에게 보낸 것은 아닐 것이라는 판단 하에 확인
recipient_count = (
    friend.groupby("send_user_id")["receive_user_id"].nunique()
)

user_friend["recipient_count"] = (
    user_friend["id"].map(recipient_count).fillna(0).astype(int)
)


user_friend.loc[
    user_friend["sent_count"] > 0,
    ["sent_count", "recipient_count"]
].describe()

# sent_count: 보낸 요청 건수
# recipient_count: 요청을 보낸 서로 다른 상대 수

,sent_count,recipient_count
count,649072.000000,649072.000000
mean,26.417986,26.250145
std,23.829836,23.228061
min,1.000000,1.000000
25%,11.000000,11.000000
50%,21.000000,21.000000
75%,35.000000,35.000000
max,1816.000000,1549.000000


In [47]:
## ④-1. 같은 상대에 대한 추가 요청이 얼마나 있는지 확인

print(friend[
    ["id", "send_user_id", "receive_user_id"]
].isna().sum())

print("요청 ID 중복:", friend["id"].duplicated().sum())

id                 0
send_user_id       0
receive_user_id    0
dtype: int64
요청 ID 중복: 0


In [48]:
senders = user_friend.loc[
    user_friend["sent_count"] > 0
].copy()

senders["extra_requests"] = (
    senders["sent_count"] - senders["recipient_count"]
)

pd.Series({
    "반복 상대 요청이 있는 사용자 수":
        senders["extra_requests"].gt(0).sum(),

    "발신자 중 해당 사용자 비율(%)":
        senders["extra_requests"].gt(0).mean() * 100,

    "추가 요청 건수":
        senders["extra_requests"].sum(),

    "전체 발신 요청 중 추가 요청 비중(%)":
        senders["extra_requests"].sum()
        / senders["sent_count"].sum() * 100
}).round(2)

반복 상대 요청이 있는 사용자 수         45048.00
발신자 중 해당 사용자 비율(%)             6.94
추가 요청 건수                  108941.00
전체 발신 요청 중 추가 요청 비중(%)         0.64
dtype: float64

-> 같은 상대에게 반복 요청한 발신자는 45,048명(6.94%)이며, 추가 요청 108,941건은 전체 발신 요청의 0.64%를 차지

### ① 요청이 많은 사용자의 기록이 특정 날짜에 몰려 있는가?

In [49]:
top_id = user_friend.loc[
    user_friend["sent_count"].idxmax(), "id"
]

top_requests = friend.loc[
    friend["send_user_id"] == top_id
]

top_requests.groupby(
    top_requests["created_at"].dt.date
)["id"].nunique().sort_values(ascending=False).head(10)

created_at
2023-06-02    1283
2023-05-31     263
2023-06-05     148
2023-06-03     102
2023-05-28      20
Name: id, dtype: int64

-> 최다 발신자는 5개 날짜에 총 1,816건을 보냈으며, 2023-06-02 하루에 1,283건(70.65%)이 집중됨

### ② 반복 요청이 일부 사용자에게 몰려 있는가?

In [50]:
senders.loc[
    senders["extra_requests"] > 0,
    "extra_requests"
].describe(percentiles=[0.5, 0.9, 0.99])

count    45048.000000
mean         2.418332
std          5.161489
min          1.000000
50%          1.000000
90%          4.000000
99%         18.000000
max        312.000000
Name: extra_requests, dtype: float64

-> 대부분은 적은 횟수를 반복하고 일부에서 큰 값이 나타남

## 2. 질문 관련 EDA

### df_questionpiece ,  df_userquestionrecord 사용

In [52]:
## ① 질문 관련 기록의 기간 확인

piece = df_questionpiece[
    ["id", "question_id", "is_voted", "is_skipped", "created_at"]
].copy()

record = df_userquestionrecord[
    ["id", "user_id", "question_piece_id", "created_at"]
].copy()

piece["created_at"] = pd.to_datetime(
    piece["created_at"], format="mixed", errors="coerce"
)

record["created_at"] = pd.to_datetime(
    record["created_at"], format="mixed", errors="coerce"
)


print("질문 단위 날짜 결측·변환 실패:", piece["created_at"].isna().sum())
print("투표 기록 날짜 결측·변환 실패:", record["created_at"].isna().sum())

pd.DataFrame({
    "질문 단위 생성 시각": piece["created_at"].agg(["min", "max"]),
    "투표 기록 생성 시각": record["created_at"].agg(["min", "max"])
})

질문 단위 날짜 결측·변환 실패: 0
투표 기록 날짜 결측·변환 실패: 0


,질문 단위 생성 시각,투표 기록 생성 시각
min,2023-04-28 12:27:22,2023-04-28 12:27:49
max,2024-05-07 11:32:30,2024-05-08 01:36:18


-> 질문 단위가 먼저 생성되고 이후 투표 기록 생성이 발생함

In [53]:
## ② 투표·스킵 상태의 기본 분포 확인

state_summary = (
    piece.groupby(
        ["is_voted", "is_skipped"],
        dropna=False
    )
    .size()
    .reset_index(name="question_piece_count")
)

state_summary["ratio_pct"] = (
    state_summary["question_piece_count"] / len(piece) * 100
).round(2)

state_summary

,is_voted,is_skipped,question_piece_count,ratio_pct
0,0,0,46789,3.7
1,1,0,1217560,96.3


-> 생성됐지만 아직 화면에 노출되지 않은 질문도 포함될 가능성도 존재

In [56]:
## ② 추가 분석 미투표 질문이 어떤 질문 세트에 속해 있는지 확인하기

import ast

piece["is_unvoted"] = (
    piece["is_voted"].eq(0) & piece["is_skipped"].eq(0)
)

set_map = df_questionset[
    ["id", "question_piece_id_list"]
].rename(columns={"id": "set_id"}).copy()

# 문자열로 저장된 질문 ID 목록을 실제 리스트로 변경
set_map["piece_id"] = (
    set_map["question_piece_id_list"].apply(ast.literal_eval)
)

# 세트 안의 질문을 한 행씩 펼치기
set_map = set_map[["set_id", "piece_id"]].explode("piece_id")
set_map = set_map.dropna(subset=["piece_id"])

# 조인할 ID의 타입 맞추기
set_map["piece_id"] = set_map["piece_id"].astype(piece["id"].dtype)

set_piece = set_map.merge(
    piece[["id", "is_voted", "is_unvoted"]],
    left_on="piece_id",
    right_on="id",
    how="left",
    validate="one_to_one"
)

print(
    "세트에 연결되지 않은 미투표 질문 수:",
    (
        piece["is_unvoted"]
        & ~piece["id"].isin(set_map["piece_id"])
    ).sum()
)

# validate="one_to_one"은 같은 질문 단위가 중복 연결되는지 확인하는 설정



set_check = set_piece.groupby("set_id").agg(
    listed_count=("piece_id", "size"),
    matched_count=("is_voted", "count"),
    voted_count=("is_voted", "sum"),
    unvoted_count=("is_unvoted", "sum")
)

set_check["type"] = "일부 투표 표시"

set_check.loc[
    set_check["voted_count"].eq(0),
    "type"
] = "투표 표시 없음"

set_check.loc[
    set_check["voted_count"].eq(set_check["listed_count"]),
    "type"
] = "전체 투표 표시"

# 목록에는 있지만 현재 piece 파일에 없는 질문이 있는 경우
set_check.loc[
    set_check["matched_count"].lt(set_check["listed_count"]),
    "type"
] = "질문 연결 누락"

set_summary = set_check.groupby("type").agg(
    set_count=("listed_count", "size"),
    unvoted_count=("unvoted_count", "sum")
)

set_summary

세트에 연결되지 않은 미투표 질문 수: 77


,set_count,unvoted_count
type,,
일부 투표 표시,359,2604
전체 투표 표시,52035,0
질문 연결 누락,101186,2858
투표 표시 없음,4125,41250


-> 157,705개 세트 중 101,186개(64.16%)에서 일부 질문이 연결되지 않아, 해당 세트의 전체 참여 상태를 판단하기 어려움

-> 전체 미투표 46,789건 중 41,250건이 투표 표시가 전혀 없는 4,125개 세트에 속하며, 세트에 연결되지 않은 미투표 질문은 77건

In [59]:
## ② 추가 분석 - 실제로 연결되지 않는 질문 ID 수와 세트당 누락 수 확인

missing = set_piece.loc[
    set_piece["id"].isna(),
    ["set_id", "piece_id"]
]

print("연결되지 않은 질문 단위 ID 수:", missing["piece_id"].nunique())
print("영향받는 세트 수:", missing["set_id"].nunique())

# 누락이 있는 세트에서, 몇 개씩 연결되지 않는지 확인
missing.groupby("set_id")["piece_id"].nunique().value_counts().sort_index()

연결되지 않은 질문 단위 ID 수: 316992
영향받는 세트 수: 101186


piece_id
1     31484
2     20831
3     14665
4     10301
5      7701
6      5530
7      4046
8      2811
9      2018
10     1799
Name: count, dtype: int64

In [57]:
## ③ 질문별로 생성된 횟수와 미투표 비율이 다른지 확인하기

question_summary = piece.groupby("question_id").agg(
    piece_count=("id", "size"),
    voted_count=("is_voted", "sum"),
    unvoted_count=("is_unvoted", "sum")
).reset_index()

question_summary["unvoted_ratio_pct"] = (
    question_summary["unvoted_count"]
    / question_summary["piece_count"] * 100
).round(2)

# 질문 내용 연결
question_summary = question_summary.merge(
    df_question[["id", "question_text"]].rename(
        columns={"id": "question_id"}
    ),
    on="question_id",
    how="left",
    validate="many_to_one"
)

# 많이 생성된 질문부터 확인
question_summary.sort_values(
    "piece_count", ascending=False
).head(10)


# 적은 건수에서 비율이 크게 보이는 경우를 줄이기 위한 탐색용 기준
MIN_PIECES = 100

question_summary.loc[
    question_summary["piece_count"] >= MIN_PIECES
].sort_values(
    ["unvoted_ratio_pct", "unvoted_count"],
    ascending=False
).head(10)

,question_id,piece_count,voted_count,unvoted_count,unvoted_ratio_pct,question_text
559,668,156,132,24,15.38,사과할 일 있는데 연락 못하고 있는 친구
723,832,173,147,26,15.03,어떤 일로 멀어졌는데 다시 관계 회복하고 싶은 사람은?
670,779,386,354,32,8.29,같이 동물원 가고 싶은 사람은?
1392,1501,298,274,24,8.05,첫사랑과 이미지가 비슷한 사람은?
824,933,356,328,28,7.87,포기가 없는 친구는 ?
1128,1237,208,192,16,7.69,화해하고 싶은 친구는?
832,941,278,257,21,7.55,사귀고 싶은 사람은?
771,880,339,314,25,7.37,아재개그에 미쳐있는 사람
1520,1629,205,190,15,7.32,화해하고 싶은 친구는?
721,830,359,333,26,7.24,겉이 차가워도 속은 촉촉한 사람


In [58]:
## ④ 사용자별 투표 횟수와 투표한 날짜 수 확인하기

# 시각을 제외하고 투표한 날짜만 남기기
record["vote_date"] = record["created_at"].dt.normalize()

user_question = record.groupby("user_id").agg(
    vote_count=("id", "nunique"),
    vote_days=("vote_date", "nunique")
)

print("투표 기록이 있는 사용자 수:", len(user_question))

user_question.describe(percentiles=[0.5, 0.9, 0.99])

user_question["day_group"] = "2일 이상 투표 기록"

user_question.loc[
    user_question["vote_days"].eq(1),
    "day_group"
] = "1일만 투표 기록"

day_summary = (
    user_question["day_group"]
    .value_counts()
    .rename("user_count")
    .to_frame()
)

day_summary["ratio_pct"] = (
    day_summary["user_count"] / len(user_question) * 100
).round(2)

day_summary

투표 기록이 있는 사용자 수: 4849


,user_count,ratio_pct
day_group,,
2일 이상 투표 기록,4698,96.89
1일만 투표 기록,151,3.11



-> 관측된 투표 참여자 대부분이 여러 날짜에 걸쳐 참여한 것

In [60]:
display(
    user_question[["vote_count", "vote_days"]]
    .describe(percentiles=[0.5, 0.9, 0.99])
)

,vote_count,vote_days
count,4849.000000,4849.000000
mean,251.094659,10.512271
std,229.023228,6.818902
min,1.000000,1.000000
50%,190.000000,9.000000
90%,525.200000,19.000000
99%,1081.600000,34.000000
max,2786.000000,79.000000
